# 01 - The Problem With The Status Quo

## 01.01 - Introductory Example

To show the problem with LLM evaluations, we will look at the [Dagstuhl 15512 Argument Quality Corpus](https://zenodo.org/records/3973285).

First download the dataset, and place it in the `datasets` folder (and/or adjust the path accordingly)

In [ ]:
from pathlib import Path

In [ ]:
path_dagstuhl = Path('../datasets') / 'dagstuhl-15512-argquality-corpus-v2' / 'dagstuhl-15512-argquality-corpus-unannotated'

In [ ]:
import pandas as pd

profiles = [
    'communist',
    'eco_justice',
    'libertarian',
    'neo_reactionary',
    'populist_nationalist',
    'religious_conservative'
]

models = [
    'anthropic/claude-sonnet-4.6',
    'openai/gpt-5.6-terra-pro',
    'google/gemma-4-31b-it',
    'x-ai/grok-4.5'
]
out = []
for model in models:
    for profile in profiles:
        path_out = Path('../data') / 'generations_anonymized' / profile / model.replace('/', '__')
        profile_generations = path_out.glob('*.txt')
        for pg in profile_generations:
            if 'global' in pg.name or 'local' in pg.name or 'emotional_appeal' in pg.name or 'overall_quality' in pg.name:
                topic, dimension1, dimension2, _id = pg.name.rsplit('_', maxsplit=3)
                dimension = f'{dimension1}_{dimension2}'
            else:
                topic, dimension, _id = pg.name.rsplit('_', maxsplit=2)
            _id = _id.split('.txt', maxsplit=1)[0]
            with open(pg, 'r', encoding='utf-8') as f:
                text = f.read()
            _, response = text.split('\n\n', maxsplit=1)
            corpus_text = response.split('How would you')[0].strip()
            path_original = path_dagstuhl / f'{topic}.csv'

            df_original = pd.read_csv(path_original, sep="\t", encoding="iso-8859-1")

            text_original = df_original[df_original['#id'] == _id]['argument'].values[0]

            text_deanonymized = text.replace(corpus_text, text_original)
            path_deanonymized = Path('../data') / 'generations_deanonymized' / profile / model.replace('/', '__') / f'{topic}_{dimension}_{_id}.txt'
            path_deanonymized.parent.mkdir(parents=True, exist_ok=True)
            with open(path_deanonymized, 'w', encoding='utf-8') as f:
                f.write(text_deanonymized)